In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
import joblib


In [10]:
df = pd.read_csv("TrainDataset2025_Preprocessed_Iter.csv")

X = df.drop(columns=["ID", "RelapseFreeSurvival (outcome)"])
y = df["RelapseFreeSurvival (outcome)"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled.shape, y.shape


((400, 118), (400,))

In [11]:
# Coarse log-scale search
C_range_coarse = np.logspace(-2, 2.7, 10)      # 0.01 → 500
gamma_range_coarse = np.logspace(-4, 0, 10)    # 1e−4 → 1

global_results = []


In [12]:
for pc in range(1, 10):  # PC = 5 → 30
    print(f"\n\n========================================")
    print(f"🔎 Testing PCA with {pc} components")
    print("========================================")

    # Step 1 — Fit PCA
    pca = PCA(n_components=pc)
    X_pca = pca.fit_transform(X_scaled)

    # Step 2 — Coarse search
    best_coarse_mae = float("inf")
    best_C_coarse = None
    best_gamma_coarse = None

    for C in C_range_coarse:
        for gamma in gamma_range_coarse:
            svr = SVR(kernel="rbf", C=C, gamma=gamma)
            mae = -cross_val_score(
                svr, X_pca, y,
                scoring="neg_mean_absolute_error", cv=5
            ).mean()

            if mae < best_coarse_mae:
                best_coarse_mae = mae
                best_C_coarse = C
                best_gamma_coarse = gamma

    print(f"Coarse best for PC={pc}:  C={best_C_coarse:.4f}, gamma={best_gamma_coarse:.6f}, MAE={best_coarse_mae:.4f}")

    # Step 3 — Fine search around coarse values
    C_range_fine = np.linspace(best_C_coarse * 0.5, best_C_coarse * 1.5, 10)
    gamma_range_fine = np.linspace(best_gamma_coarse * 0.5, best_gamma_coarse * 1.5, 10)

    best_fine_mae = float("inf")
    best_C_fine = None
    best_gamma_fine = None

    for C2 in C_range_fine:
        for g2 in gamma_range_fine:

            svr = SVR(kernel="rbf", C=C2, gamma=g2)
            mae2 = -cross_val_score(
                svr, X_pca, y,
                scoring="neg_mean_absolute_error", cv=5
            ).mean()

            if mae2 < best_fine_mae:
                best_fine_mae = mae2
                best_C_fine = C2
                best_gamma_fine = g2

    print(f"Fine best for PC={pc}:  C={best_C_fine:.4f}, gamma={best_gamma_fine:.6f}, MAE={best_fine_mae:.4f}")

    # Store global results
    global_results.append({
        "PC": pc,
        "MAE": best_fine_mae,
        "C": best_C_fine,
        "gamma": best_gamma_fine,
        "pca_model": pca
    })




🔎 Testing PCA with 1 components
Coarse best for PC=1:  C=501.1872, gamma=0.000278, MAE=23.0610
Fine best for PC=1:  C=640.4059, gamma=0.000263, MAE=23.0404


🔎 Testing PCA with 2 components
Coarse best for PC=2:  C=1.2271, gamma=1.000000, MAE=23.0571
Fine best for PC=2:  C=1.8407, gamma=1.500000, MAE=23.0134


🔎 Testing PCA with 3 components
Coarse best for PC=3:  C=13.5936, gamma=0.000100, MAE=22.9917
Fine best for PC=3:  C=15.8592, gamma=0.000117, MAE=22.9848


🔎 Testing PCA with 4 components
Coarse best for PC=4:  C=13.5936, gamma=0.000100, MAE=22.9998
Fine best for PC=4:  C=17.3696, gamma=0.000106, MAE=22.9719


🔎 Testing PCA with 5 components
Coarse best for PC=5:  C=45.2434, gamma=1.000000, MAE=22.9928
Fine best for PC=5:  C=52.7840, gamma=1.500000, MAE=22.8559


🔎 Testing PCA with 6 components
Coarse best for PC=6:  C=45.2434, gamma=1.000000, MAE=22.7874
Fine best for PC=6:  C=52.7840, gamma=1.277778, MAE=22.7549


🔎 Testing PCA with 7 components
Coarse best for PC=7:  C=45.24

In [13]:
pca_results_df = pd.DataFrame(global_results).sort_values("MAE")
pca_results_df


,PC,MAE,C,gamma,pca_model
7,8,22.563703,52.784006,0.339416,PCA(n_components=8)
8,9,22.637023,52.784006,0.299484,PCA(n_components=9)
6,7,22.682072,47.756958,0.459210,PCA(n_components=7)
5,6,22.754917,52.784006,1.277778,PCA(n_components=6)
4,5,22.855900,52.784006,1.500000,PCA(n_components=5)
3,4,22.971875,17.369554,0.000106,PCA(n_components=4)
2,3,22.984806,15.859158,0.000117,PCA(n_components=3)
1,2,23.013443,1.840688,1.500000,PCA(n_components=2)
0,1,23.040425,640.405910,0.000263,PCA(n_components=1)


In [14]:
best_pca_model = pca_results_df.iloc[0]
best_pca_model


PC                             8
MAE                    22.563703
C                      52.784006
gamma                   0.339416
pca_model    PCA(n_components=8)
Name: 7, dtype: object

In [15]:
final_pc = best_pca_model["PC"]
final_C = best_pca_model["C"]
final_gamma = best_pca_model["gamma"]

# Refit PCA on full data
pca_final = PCA(n_components=final_pc)
X_pca_final = pca_final.fit_transform(X_scaled)

# Train final SVR
svr_final = SVR(kernel="rbf", C=final_C, gamma=final_gamma)
svr_final.fit(X_pca_final, y)


SVR(C=np.float64(52.78400571052874), gamma=np.float64(0.3394157349148813))

In [16]:
joblib.dump(pca_final, "pca_transform_best.pkl")
joblib.dump(scaler, "svm_scaler_pca_best.pkl")
joblib.dump(svr_final, "svr_model_pca_best.pkl")

print("Saved PCA + SVR optimized models!")


Saved PCA + SVR optimized models!
